# ChatAPG – eenvoudige tabelspike

## Het idee in één zin

Lees een Markdown-tabel, geef iedere cel een uniek node-ID, splits een grote tabel alleen tussen volledige rijen en bewaar per chunk welke nodes erin zitten. Daardoor kan een citation de juiste, leesbare tabelchunk openen.

## De vier stappen

1. Markdown omzetten naar kolommen en rijen.
2. Iedere cel een node-ID geven.
3. Grote tabellen per groep volledige rijen chunken; headers steeds herhalen.
4. Bij een citation de chunk zoeken die het node-ID bevat.

In [ ]:
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(value):
        print(value)

## 1. Voorbeeldinput

De data is synthetisch. Een lege waarde blijft leeg en een letterlijk `-` blijft een streepje.

In [ ]:
MARKDOWN_TABLE = '''| Regeling | OP 2026 | RP 2026 | Totaal 2026 |
|---|---:|---:|---:|
| Regeling A | 22,00% | 3,93% | 25,93% |
| Regeling B | 22,00% | 3,73% | 25,73% |
| Regeling C | 24,00% |  | 24,00% |
| Regeling D | 21,00% | - | 21,00% |
| Regeling E | 20,00% | 3,20% | 23,20% |
| Regeling F | 23,00% | 3,10% | 26,10% |
| Regeling G | 19,00% | 2,90% | 21,90% |
| Regeling H | 25,00% | 4,00% | 29,00% |'''

display(Markdown(MARKDOWN_TABLE))

## 2. De volledige spikecode

De code hieronder doet alleen wat voor de demonstratie nodig is. `rows_per_chunk` maakt het splitsgedrag zichtbaar; in productie wordt dit vervangen door het echte tokenbudget.

In [ ]:
def parse_markdown_table(markdown):
    lines = [line.strip() for line in markdown.splitlines() if line.strip()]
    split = lambda line: [cell.strip() for cell in line.strip('|').split('|')]
    headers = split(lines[0])
    rows = [split(line) for line in lines[2:]]
    if any(len(row) != len(headers) for row in rows):
        raise ValueError('Een rij heeft niet hetzelfde aantal cellen als de header')
    return headers, rows

def render_markdown(headers, rows):
    clean = lambda value: str(value).replace('|', '\\|').replace('\n', ' ')
    lines = [
        '| ' + ' | '.join(map(clean, headers)) + ' |',
        '| ' + ' | '.join(['---'] * len(headers)) + ' |',
    ]
    lines += ['| ' + ' | '.join(clean(value or '—') for value in row) + ' |' for row in rows]
    return '\n'.join(lines)

def make_nodes_and_chunks(markdown, rows_per_chunk=3):
    headers, rows = parse_markdown_table(markdown)
    nodes = [
        {
            'id': f'r{row_index}-c{column_index}',
            'row_index': row_index,
            'column_index': column_index,
            'column_name': headers[column_index],
            'text': value,
        }
        for row_index, row in enumerate(rows)
        for column_index, value in enumerate(row)
    ]

    chunks = []
    for start in range(0, len(rows), rows_per_chunk):
        end = min(start + rows_per_chunk, len(rows))
        chunks.append({
            'chunk_index': len(chunks),
            'row_start': start,
            'row_end': end - 1,
            'node_ids': [node['id'] for node in nodes if start <= node['row_index'] < end],
            'content': render_markdown(headers, rows[start:end]),
        })

    return nodes, chunks

## 3. Grote tabel opdelen zonder een rij kapot te maken

In [ ]:
nodes, chunks = make_nodes_and_chunks(MARKDOWN_TABLE, rows_per_chunk=3)

print('Aantal rijen:', len(nodes) // 4)
print('Aantal nodes:', len(nodes))
print('Aantal chunks:', len(chunks))
print('Rijbereiken:', [(chunk['row_start'], chunk['row_end']) for chunk in chunks])

for chunk in chunks:
    print(f"\nChunk {chunk['chunk_index']} – rijen {chunk['row_start']} t/m {chunk['row_end']}")
    display(Markdown(chunk['content']))

## 4. Citation-click simuleren

Stel dat het antwoord verwijst naar één cel. Het node-ID van die cel bepaalt welke chunk de frontend moet openen.

In [ ]:
clicked_node_id = 'r4-c2'
selected_chunk = next(chunk for chunk in chunks if clicked_node_id in chunk['node_ids'])
selected_node = next(node for node in nodes if node['id'] == clicked_node_id)

print('Geklikte node:', selected_node)
print('Frontend toont chunk:', selected_chunk['chunk_index'])
display(Markdown(selected_chunk['content']))

## 5. Snelle controles

In [ ]:
all_ids = [node['id'] for node in nodes]
chunk_ids = [node_id for chunk in chunks for node_id in chunk['node_ids']]

assert len(all_ids) == len(set(all_ids))
assert chunk_ids == all_ids
assert all('| Regeling | OP 2026 | RP 2026 | Totaal 2026 |' in chunk['content'] for chunk in chunks)
assert next(node for node in nodes if node['id'] == 'r2-c2')['text'] == ''
assert next(node for node in nodes if node['id'] == 'r3-c2')['text'] == '-'
assert clicked_node_id in selected_chunk['node_ids']

print('SIMPLE_TABLE_SPIKE=PASS')

## Zo leg je dit uit

> De tabel wordt eerst gelezen als kolommen en rijen. Iedere cel krijgt een node-ID met zijn rij, kolom en waarde. Bij een grote tabel groeperen we alleen volledige rijen en herhalen we de headers. Een citation bevat een node-ID; daarmee vinden we de chunk die React als Markdown kan tonen.

## Wat deze eenvoudige spike nog niet doet

- Automatisch tabellen vinden tussen gewone documenttekst.
- Complexe HTML-, PDF- of `rowspan`/`colspan`-tabellen verwerken.
- Echte tokenlimieten gebruiken.
- De nodes al naar de vectordatabase of bestaande citationketen schrijven.

Dat zijn vervolgstappen nadat bevestigd is dat deze simpele node- en chunkrichting past bij de pilot.